# Геокодирование через Nominatim (OSM) для сайта ГВС-Ярославль

Скрипт получает координаты адресов из `data.json` и сохраняет результат в `geo.json`.

**Как пользоваться:**
1. Запустите ячейки по порядку.
2. В ШАГ 2 загрузите файл `data.json`.
3. ШАГ 6 (опционально) — загрузите существующий `geo.json`, чтобы продолжить прерванный сеанс.
4. После геокодирования скачайте `geo.json` и положите его рядом с `index.html`.

In [ ]:
# ШАГ 1 — Установка библиотек
!pip install requests --quiet
print('✅ Готово')

In [ ]:
# ШАГ 2 — Импорты, настройки и загрузка data.json
import requests, json, time, os, re
from google.colab import files

GEO_FILE = 'geo.json'
DELAY    = 1.1   # секунд между запросами (политика Nominatim: max 1 RPS)

NOMINATIM_URL = 'https://nominatim.openstreetmap.org/search'
HEADERS = {'User-Agent': 'gvs-yaroslavl/1.0 (github.com/Aoneti/gvs-yaroslavl)'}

print('📂 Выберите файл data.json')
uploaded = files.upload()
data_filename = list(uploaded.keys())[0]
with open(data_filename, encoding='utf-8') as f:
    data = json.load(f)

addresses = sorted(set(item['address'] for item in data))
print(f'\n✅ Загружен {data_filename}: {len(data)} записей, {len(addresses)} уникальных адресов')

In [ ]:
# ШАГ 3 — Географические границы и справочник посёлков
YAROSLAVL_BBOX = {
    'lat_min': 57.52, 'lat_max': 57.73,
    'lng_min': 39.74, 'lng_max': 40.08,
}
YAROSLAVL_DISTRICT_BBOX = {
    'lat_min': 57.30, 'lat_max': 57.90,
    'lng_min': 39.40, 'lng_max': 40.40,
}

RURAL_SETTLEMENTS = {
    'дубки':         'Дубки',
    'ивняки':        'Ивняки',
    'щедрино':       'Щедрино',
    'карабиха':      'Карабиха',
    'толбухино':     'Толбухино',
    'лесная поляна': 'Лесная Поляна',
    'нагорный':      'Нагорный',
}

def get_locality(addr_lower: str) -> str:
    for keyword, locality in RURAL_SETTLEMENTS.items():
        if keyword in addr_lower:
            return locality
    return 'Ярославль'

def is_rural(addr_lower: str) -> bool:
    return any(k in addr_lower for k in RURAL_SETTLEMENTS)

_STREET_START_RE = re.compile(
    r'\b(ул\.?|улица|пер\.?|переулок|пр-д\.?|проезд|пр-кт\.?|проспект'
    r'|набережная|шоссе|бульвар|площадь|микрорайон|б-р\.?)\b',
    re.IGNORECASE
)

def extract_street_part(addr: str) -> str:
    m = _STREET_START_RE.search(addr)
    return addr[m.start():].strip() if m else addr

print('✓ Географические границы и справочник загружены')

In [ ]:
# ШАГ 4 — Функции валидации
def is_valid_location(lat: float, lng: float, rural: bool) -> bool:
    bbox = YAROSLAVL_DISTRICT_BBOX if rural else YAROSLAVL_BBOX
    return (
        bbox['lat_min'] <= lat <= bbox['lat_max'] and
        bbox['lng_min'] <= lng <= bbox['lng_max']
    )

def display_name_ok(display_name: str, rural: bool) -> bool:
    dn = display_name.lower()
    if rural:
        return (
            ('ярославский' in dn and ('район' in dn or 'округ' in dn))
            or 'ярославская область' in dn
        )
    return (
        'ярославль' in dn
        and 'ярославский район' not in dn
        and 'ярославский муниципальный округ' not in dn
    )

def validate_result(result: dict, addr_lower: str) -> bool:
    rural    = is_rural(addr_lower)
    lat, lng = result['lat'], result['lng']
    dn       = result.get('full_name', '')
    coord_ok = is_valid_location(lat, lng, rural)
    name_ok  = display_name_ok(dn, rural)
    if not coord_ok:
        print(f'      ⛔ bbox fail: {lat:.4f}, {lng:.4f} — {dn[:60]}')
    elif not name_ok:
        print(f'      ⛔ name fail: {dn[:80]}')
    return coord_ok and name_ok

print('✓ Функции валидации загружены')

In [ ]:
# ШАГ 5 — Нормализация адресов для Nominatim
def normalize_address(raw: str) -> str:
    s = raw.strip()
    s = re.sub(r'\b(\d+)-(ой|ый|ий|ей)\b', r'\1-й', s, flags=re.IGNORECASE)
    s = re.sub(r'\b(\d+)-(ая|яя)\b',        r'\1-я', s, flags=re.IGNORECASE)
    s = re.sub(r'\b(\d+)-(ое|ее)\b',        r'\1-е', s, flags=re.IGNORECASE)
    s = re.sub(
        r'д\.?\s*(\d+\w*)[,\s]+к(?:орп?)?\.?\s*(\d+)',
        r'\1к\2', s, flags=re.IGNORECASE
    )
    s = re.sub(r'\bк(?:орп?)?\.?\s*(\d+)', r'к\1', s, flags=re.IGNORECASE)
    street_types = [
        (r'\bпр-кт\.?\b',    'проспект'),
        (r'\bпр-д\.?\b',     'проезд'),
        (r'\bпер\.?\b',      'переулок'),
        (r'\bпл\.?\b',       'площадь'),
        (r'\bб-р\.?\b',      'бульвар'),
        (r'\bнаб\.?\b',      'набережная'),
        (r'\bмкр\.?\b',      'микрорайон'),
        (r'\bкв-л\.?\b',     'квартал'),
        (r'\bш\.?\b',        'шоссе'),
        (r'\bул\.?\b',       'улица'),
        (r'\bпр\.(?!\s*-)',  'проспект'),
    ]
    for pattern, replacement in street_types:
        s = re.sub(pattern, replacement, s, flags=re.IGNORECASE)
    s = re.sub(r'\bулица\.', 'улица', s)
    s = re.sub(r'\bпроспект\.', 'проспект', s)
    s = re.sub(r'\bд\.?\s*(?=\d)', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\bстр\.?\s*(\d+)', r'с\1', s, flags=re.IGNORECASE)
    s = re.sub(r'\bлит\.?\s*\w*',   '',     s, flags=re.IGNORECASE)
    s = re.sub(r'\b(посёлок|поселок|пос\.?)\s+', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s{2,}', ' ', s).strip(' ,.')
    return s

test_cases = [
    '2-ой Брагинский пр-д д. 3',
    'пр-кт Авиаторов, д.74',
    'ул Красноборская, д.38, корп.3',
    'ул. Шоссейная 1-я, д.32, корп.2',
    'посёлок Дубки ул. Ленина д. 1',
    'посёлок Ивняки ул. Светлая д. 2',
]
print('Проверка нормализации:')
for t in test_cases:
    print(f'  {t}  →  {normalize_address(t)}')

In [ ]:
# ШАГ 6 — Функция геокодирования
def geocode_address(addr: str) -> dict | None:
    addr_lower = addr.lower()
    rural      = is_rural(addr_lower)
    locality   = get_locality(addr_lower)

    if rural:
        street_raw  = extract_street_part(addr)
        street_norm = normalize_address(street_raw)
        street_only = re.sub(r'\s+\d+\S*\s*$', '', street_norm).strip()
        strategies = [
            {'street': street_norm,  'city': locality, 'country': 'Россия',
             'format': 'jsonv2', 'limit': 3},
            {'q': f'{locality}, {street_norm}',  'format': 'jsonv2', 'limit': 3},
            {'q': f'{locality}, {street_raw}',   'format': 'jsonv2', 'limit': 3},
            {'q': f'{locality}, {street_only}',  'format': 'jsonv2', 'limit': 3},
        ]
    else:
        norm        = normalize_address(addr)
        street_only = re.sub(r'[,\s]+\d+\S*\s*$', '', norm).strip()
        strategies = [
            {'street': norm,          'city': locality, 'country': 'Россия',
             'format': 'jsonv2', 'limit': 3},
            {'q': f'{locality}, {norm}',  'format': 'jsonv2', 'limit': 3},
            {'q': f'{locality}, {addr}',  'format': 'jsonv2', 'limit': 3},
            {'q': f'{locality}, {street_only}', 'format': 'jsonv2', 'limit': 3},
        ]

    for i, params in enumerate(strategies):
        if i > 0:
            time.sleep(DELAY)
        try:
            r = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
            if r.status_code != 200:
                continue
            candidates = r.json()
            if not candidates:
                continue
            for candidate in candidates:
                lat = round(float(candidate['lat']), 6)
                lng = round(float(candidate['lon']), 6)
                dn  = candidate.get('display_name', '')
                norm_used = street_norm if rural else normalize_address(addr)
                result = {
                    'lat':         lat,
                    'lng':         lng,
                    'full_name':   dn,
                    'strategy':    i + 1,
                    'norm':        norm_used,
                    'approximate': (i + 1 == 4),
                }
                if validate_result(result, addr_lower):
                    return result
        except Exception as e:
            print(f'    ⚠️  Стратегия {i+1}: {e}')

    return None

print('✓ Функция геокодирования загружена')

In [ ]:
# ШАГ 7 (опционально) — Загрузить существующий geo.json для продолжения
# Раскомментируйте строки ниже, если хотите продолжить прерванный сеанс!

# uploaded_geo = files.upload()
# geo_filename = list(uploaded_geo.keys())[0]
# import shutil; shutil.copy(geo_filename, GEO_FILE)
# print('Файл загружен:', geo_filename)

print('ℹ️  Ячейка для загрузки кэша (опционально). Раскомментируйте при необходимости.')

In [ ]:
# ШАГ 8 — Инициализация кэша и очередь обработки
geo_db = {}
if os.path.exists(GEO_FILE):
    with open(GEO_FILE, encoding='utf-8') as f:
        geo_db = json.load(f)

    if not isinstance(geo_db, dict):
        print('⚠️  geo.json повреждён (не dict), начинаем с нуля')
        geo_db = {}
    else:
        done = sum(1 for v in geo_db.values() if v)
        print(f'  ♻️  Кэш: {len(geo_db)} записей, {done} с координатами')

        failed = [k for k, v in geo_db.items() if v is None]
        for k in failed:
            del geo_db[k]
        if failed:
            print(f'  🔄 Сброшено {len(failed)} неудачных — повторим')

        invalid = []
        for k, v in list(geo_db.items()):
            if v and not is_valid_location(v['lat'], v['lng'], is_rural(k.lower())):
                invalid.append(k)
                del geo_db[k]
        if invalid:
            print(f'  🔄 Сброшено {len(invalid)} адресов вне bbox — перегеокодируем:')
            for a in invalid:
                print(f'     • {a}')
else:
    print('  Кэш не найден, начинаем с нуля')

to_process = [a for a in addresses if a not in geo_db]
print(f'  Осталось обработать: {len(to_process)}')
print(f'  Ожидаемое время: ~{len(to_process) * DELAY * 2 / 60:.0f} мин')
# × 2 — средняя поправка на retry через стратегии

In [ ]:
# ШАГ 9 — Геокодирование
# ⚠️ Не прерывайте — кэш сохраняется после каждого адреса.
# Если сессия прервётся, скачайте geo.json и загрузите через ШАГ 7.
print('Геокодирование...')
print('-' * 58)

ok_cached   = sum(1 for v in geo_db.values() if v)
ok_session   = 0
fail_session = 0

for i, addr in enumerate(to_process):
    result = geocode_address(addr)

    if result:
        geo_db[addr] = result
        ok_session += 1
        strat  = result['strategy']
        approx = result.get('approximate', False)
        if approx:
            marker = '🔸'
        elif strat == 1:
            marker = '✅'
        else:
            marker = '🔶'
        approx_tag = ' [~приблизительно]' if approx else ''
        print(f'  [{i+1}/{len(to_process)}] {marker}(стр.{strat}){approx_tag} '
              f'{addr[:38]}  →  {result["norm"]}')
    else:
        geo_db[addr] = None
        fail_session += 1
        norm_used = normalize_address(extract_street_part(addr) if is_rural(addr.lower()) else addr)
        print(f'  [{i+1}/{len(to_process)}] ❌  {addr}  →  {norm_used}')

    with open(GEO_FILE, 'w', encoding='utf-8') as f:
        json.dump(geo_db, f, ensure_ascii=False, indent=2)

    time.sleep(DELAY)

print('\n' + '=' * 58)
total_ok = ok_cached + ok_session
print(f'✅ Успешно: {total_ok} из {len(addresses)}')
print(f'   из кэша: {ok_cached}  |  получено сейчас: {ok_session}')
if fail_session:
    print(f'❌ Не найдено в этом сеансе: {fail_session}')
    not_found = [a for a, v in geo_db.items() if not v]
    for a in not_found:
        norm_used = normalize_address(extract_street_part(a) if is_rural(a.lower()) else a)
        print(f'    • {a}  →  {norm_used}')

s = [0, 0, 0, 0]
approx_count = 0
for v in geo_db.values():
    if v and v.get('strategy'):
        idx = v['strategy'] - 1
        if 0 <= idx < 4:
            s[idx] += 1
    if v and v.get('approximate'):
        approx_count += 1
print(f'\n  Стр.1 структурир.+норм.:   {s[0]}')
print(f'  Стр.2 свободный+норм.:     {s[1]}')
print(f'  Стр.3 свободный+ориг.:     {s[2]}')
print(f'  Стр.4 только улица (~):    {s[3]}')
if approx_count:
    print(f'\n  ⚠️  Приблизительных координат: {approx_count} — на карте будут помечены особо.')

In [ ]:
# ШАГ 10 — Скачать результат
print(f'Скачиваю {GEO_FILE}…')
files.download(GEO_FILE)
print('✓ Готово! Положите geo.json рядом с index.html в репозитории.')